# 🧪 W5-D2 概念实验：ToT 搜索、剪枝，与图上的推理

> 配套阅读：`ima/第5周-Day2-Tree-of-Thought与图推理.md`（ToT 四步骤、BFS/DFS 策略、业务场景在那边）
>
> 本 notebook 回答三个问题：
> 1. 同一棵"思维树"，暴力枚举 / ToT-BFS（beam 剪枝）/ 贪心 DFS 的**评估开销与解质量**差多少？
> 2. 剪枝的风险：启发式打分有噪声时，越贪心越容易**错过全局最优**吗？
> 3. 图推理：贪心"只看眼前最短边"为什么会绕远路？（Dijkstra 对照）

实验环境：随机生成的思维树 + 加权图，纯 numpy / 标准库。

## 实验 1：三种搜索策略跑 300 棵随机思维树

树：分支 B=3、深度 D=4 → 81 个叶子，叶子价值 ~ N(0,1)。
搜索时代价是**评估次数**（每次让 LLM 给候选打分 = 1 次评估），打分带噪声（σ=0.2）。
- 暴力枚举：全部 81 个叶子都评一遍
- ToT-BFS（beam）：逐层展开，每层只保留启发式最好的 beam 个前缀 —— **这就是剪枝**
- 贪心 DFS：每层只走当前最好的一支，一条路走到黑

In [ ]:
import numpy as np

B, D = 3, 4   # 分支 3、深度 4 → 3^4 = 81 个叶子

def make_leaves(seed):
    return np.random.default_rng(seed).normal(0, 1, (B,) * D)

def h_eval(leaves, prefix, rng, sigma=0.2):
    """启发式评估 = 前缀子树的真实最优值 + 打分噪声（模拟 LLM 打分不完美）"""
    return leaves[prefix].max() + rng.normal(0, sigma)

def search_exhaustive(leaves):
    return float(leaves.max()), leaves.size

def search_tot_beam(leaves, rng, beam, sigma=0.2):
    """ToT-BFS：逐层展开，只保留启发式最好的 beam 个前缀（剪枝）"""
    prefixes, evals = [()], 0
    for _ in range(D):
        children = []
        for pre in prefixes:
            for b in range(B):
                children.append((h_eval(leaves, pre + (b,), rng, sigma), pre + (b,)))
                evals += 1
        children.sort(key=lambda x: -x[0])
        prefixes = [idx for _, idx in children[:beam]]
    return max(float(leaves[idx]) for idx in prefixes), evals

def search_dfs_greedy(leaves, rng, sigma=0.2):
    """贪心 DFS：每层只走当前启发式最好的分支"""
    idx, evals = (), 0
    while len(idx) < D:
        scores = [(h_eval(leaves, idx + (b,), rng, sigma), b) for b in range(B)]
        evals += B
        idx = idx + (max(scores)[1],)
    return float(leaves[idx]), evals

trees = [make_leaves(2000 + t) for t in range(300)]
opts = np.array([float(lv.max()) for lv in trees])

configs = {
    "暴力枚举":        lambda lv, r: search_exhaustive(lv),
    "ToT-BFS beam=2": lambda lv, r: search_tot_beam(lv, r, 2),
    "ToT-BFS beam=3": lambda lv, r: search_tot_beam(lv, r, 3),
    "贪心DFS":        lambda lv, r: search_dfs_greedy(lv, r),
}
r_eval = np.random.default_rng(99)
res = {}
for name, fn in configs.items():
    bests, evals_list = [], []
    for lv in trees:
        v, e = fn(lv, r_eval)
        bests.append(v); evals_list.append(e)
    bests = np.array(bests)
    res[name] = (float(bests.mean()), float(np.mean(evals_list)), float((bests >= opts - 1e-9).mean()))
    print(f"{name:16s} 平均叶子价值 {res[name][0]:+.3f} | 平均评估 {res[name][1]:5.1f} 次 | 命中全局最优 {res[name][2]:.0%}")

## 实验 2：把"省了多少算力、丢了多少质量"画出来

横轴：平均评估次数（= 打分调用 = Token 花销）。
纵轴：命中最优解的比例。理想策略在左上角：又便宜又准。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

names = list(res)
x = np.arange(len(names))
fig, ax1 = plt.subplots(figsize=(8, 4.2))
ax1.bar(x, [res[n][1] for n in names], color="#8ecae6", label="平均评估节点数")
ax1.set_xticks(x, names)
ax1.set_ylabel("平均评估节点数（= 打分调用）")
ax2 = ax1.twinx()
ax2.plot(x, [res[n][2] * 100 for n in names], "ro-", label="命中最优比例")
ax2.set_ylabel("命中全局最优 (%)", color="r")
ax1.set_title("ToT 用 beam 剪枝：以少量质量损失换大幅算力节省")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="center right", fontsize=8)
plt.tight_layout(); plt.show()

b2, b3, dfs = res["ToT-BFS beam=2"], res["ToT-BFS beam=3"], res["贪心DFS"]
print(f"beam=2 只用 21/81 ≈ 1/4 的评估，就命中 {b2[2]:.0%} 的最优解；beam=3 命中 {b3[2]:.0%}。")
print(f"贪心 DFS 最便宜（{dfs[1]:.0f} 次评估）但只命中 {dfs[2]:.0%} —— 省钱与质量按业务代价权衡。")

## 实验 3：剪枝的代价——打分噪声越大，越贪心越危险

LLM 的启发式打分天然有噪声。固定树，变化噪声 σ 与 beam 宽度，
统计 300 棵树上**找到全局最优解的概率**。σ=0 等于完美打分（剪枝无风险）。

In [ ]:
import numpy as np
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

sigmas = [0.0, 0.2, 0.5, 1.0]
r = np.random.default_rng(77)

plt.figure(figsize=(7, 4))
for beam in (1, 2, 3):
    hit_rates = []
    for s in sigmas:
        hits = 0
        for lv in trees:
            v, _ = search_tot_beam(lv, r, beam, s)
            hits += v >= float(lv.max()) - 1e-9
        hit_rates.append(hits / len(trees) * 100)
    plt.plot(sigmas, hit_rates, "o-", label=f"beam={beam}" + ("（纯贪心）" if beam == 1 else ""))
plt.xlabel("启发式打分噪声 σ")
plt.ylabel("命中全局最优的比例 (%)")
plt.title("剪枝风险：打分越不准、保留越少，越容易错过最优分支")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("对应到真实 ToT：让 LLM 打分时给出理由/多次打分取平均（降噪）、")
print("或放宽 beam 宽度，都是在买回'剪掉好分支'的风险。")

## 实验 4：图推理——贪心"只看眼前"vs Dijkstra 全局最优

把"推理路径"建成加权图：节点是状态，边权是一步代价。
- 贪心：站在当前节点，永远走**相邻最短边**
- Dijkstra：维护"从起点到每个点的最短距离"，保证全局最优

图里故意埋一个陷阱：A→B 的边短，但走过去之后绕远。

In [ ]:
import heapq

graph = {           # 经典陷阱图：局部便宜的 A→B 通往昂贵的后半程
    "A": {"B": 2, "C": 4},
    "B": {"C": 6, "D": 9},
    "C": {"D": 3, "E": 6},
    "D": {"E": 2},
    "E": {},
}

def dijkstra(graph, src, dst):
    dist, prev, pq, seen = {src: 0}, {}, [(0, src)], set()
    while pq:
        d, u = heapq.heappop(pq)
        if u in seen:
            continue
        seen.add(u)
        for v, w in graph[u].items():
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, u
                heapq.heappush(pq, (d + w, v))
    path, node = [dst], dst
    while node != src:
        node = prev[node]; path.append(node)
    return dist[dst], "-".join(path[::-1])

def greedy(graph, src, dst):
    """每步只看相邻边最短——图推理里的'直线思维'"""
    path, node, cost = [src], src, 0
    while node != dst:
        nxt = min(graph[node], key=graph[node].get)
        cost += graph[node][nxt]; path.append(nxt); node = nxt
    return cost, "-".join(path)

opt = dijkstra(graph, "A", "E")
grd = greedy(graph, "A", "E")
print(f"Dijkstra（全局最优）: 代价 {opt[0]:>2}, 路径 {opt[1]}")
print(f"贪心（只看眼前）    : 代价 {grd[0]:>2}, 路径 {grd[1]}")
print()
print("贪心在 A 处被 2<4 的便宜边骗走，最后多花 %.0f%% 代价。" % ((grd[0]/opt[0]-1)*100))
print("ToT 的'往前多看几层再决定'，本质就是给贪心补上全局信息。")

## 结论

| 策略 | 评估次数 | 解质量 | 适用 |
|---|---|---|---|
| 暴力枚举 | 最多 | 最优保证 | 候选少、每步可精确评分 |
| ToT-BFS(beam) | 中 | 接近最优 | 分支多、打分可靠 |
| 贪心 DFS | 最少 | 局部最优风险高 | 预算紧、可接受次优 |

剪枝省算力，但**打分噪声 + 过度剪枝 = 丢掉好分支**（实验 3）；
图上贪心短视会被"便宜的下一步"骗（实验 4）。

→ 深入阅读：`ima/第5周-Day2-Tree-of-Thought与图推理.md`（ToT 四步骤、自洽性、知识图谱推理业务案例）